In [ ]:
#@title ⚙️ Setup — choose how to provide the inputs (RUN ME FIRST)
#@markdown Pick how the input files for this step are provided:
#@markdown - **Mobile mode**: the files are downloaded automatically from the GitHub repo — no manual upload, works on phones/tablets.
#@markdown - **Manual upload**: you upload the files yourself in the cells below.
input_mode = "Mobile mode (auto-download from repo)" #@param ["Mobile mode (auto-download from repo)", "Manual upload"]

import os, urllib.request
REPO_RAW = "https://raw.githubusercontent.com/biochorl/Nanobody_de_novo_design/main"

# Inputs needed by THIS step.  (some links are FACSIMILE placeholders until the files
# are added to the repo — they will start working once they are committed)
INPUT_FILES = [('/content/PIPPack_outputs.zip', '/Intermediate_inputs/PIPPack_outputs.zip', 'facsimile'), ('/content/nanobody_redesign.fasta', '/Intermediate_inputs/nanobody_redesign.fasta', 'real')]

def fetch_inputs(files):
    ok = 0
    for dst, rel, kind in files:
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        url = REPO_RAW + rel
        try:
            urllib.request.urlretrieve(url, dst)
            print(f"✅ {os.path.basename(dst):<28} ← {url}  [{kind}]")
            ok += 1
        except Exception as e:
            tag = "facsimile not in repo yet" if kind == "facsimile" else "ERROR"
            print(f"⚠️  {os.path.basename(dst):<28} could not be downloaded ({tag}): {e}")
    return ok

if input_mode.startswith("Mobile"):
    print("📱 Mobile mode — downloading this step's inputs from the repo:\n")
    n = fetch_inputs(INPUT_FILES)
    print(f"\n→ {n}/{len(INPUT_FILES)} file(s) ready in /content. "
          "Skip the manual-upload cells and point the path fields to these files.")
else:
    print("🖱️ Manual mode — upload your files in the cells below as usual.")


# **AntiFold for nanobody CDRs (re)design**
## Overview
####[Antifold](https://github.com/oxpig/AntiFold) ([paper link](https://academic.oup.com/bioinformaticsadvances/article/5/1/vbae202/8090019)) is a model trained to reconstruct the sequence of complementarity-determining regions (CDRs) of antibody- or nanobody-antigen complexes. It has a superior accuracy on CDRs with respect to ProteinMPNN
---

In [ ]:
# @title Install AntiFold and Dependencies
# @markdown Run this cell first to install all required packages

# Install Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -q
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!bash ./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# Accept Conda Terms of Service using the full path
!/usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!/usr/local/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Initialize conda
!source /usr/local/bin/activate
!/usr/local/bin/conda init bash

# Set up conda environment
!/usr/local/bin/conda create --name antifold python=3.10 -y

# Use the full path to the conda environment to avoid activation issues
!/usr/local/envs/antifold/bin/pip install torch==2.2.0 --index-url https://download.pytorch.org/whl/cpu

# Clone and install AntiFold
!git clone https://github.com/oxpig/AntiFold
%cd AntiFold
!/usr/local/envs/antifold/bin/pip install .

# Install additional dependencies for the notebook
!/usr/local/envs/antifold/bin/pip install biopython pandas ipywidgets

# Create necessary directories
!mkdir -p data/pdbs
!mkdir -p output

print("Installation completed successfully!")

## Upload Files and Run AntiFold
Upload the ZIP containing your PDB designs (e.g. from PIPPack) and the `nanobody_redesign.fasta` generated in step 1. AntiFold will automatically detect the `X` residues and mutate strictly those positions.

In [ ]:
# @title Run AntiFold Redesign
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os, zipfile, subprocess, glob
import pandas as pd
from google.colab import files

display(HTML("<h2>Nanobody Design Parameters</h2>"))
num_sequences = widgets.IntSlider(description="Number of sequences:", min=1, max=100, value=10)
seed_value = widgets.Text(description="Random seed:", value="5748")
temperature = widgets.FloatSlider(description="Sampling temperature:", min=0.1, max=1.0, step=0.1, value=0.3)
use_json_annotation = widgets.Checkbox(value=True, description='Use dynamic CDR JSON (recommended)')
display(num_sequences, seed_value, temperature, use_json_annotation)

display(HTML("<h3>1. Upload nanobody_redesign.fasta (optional fallback if no JSON)</h3>"))
upload_fasta = widgets.FileUpload(description="Choose FASTA", accept='.fasta', multiple=False)
display(upload_fasta)

display(HTML("<h3>2. Upload PIPPack_outputs.zip (PDBs + cdrs_annotation.json)</h3>"))
upload_pdbs = widgets.FileUpload(description="Choose ZIP/PDB", accept='.zip,.pdb,.json', multiple=True)
display(upload_pdbs)

run_button = widgets.Button(description="Run AntiFold", button_style='success')
output_area = widgets.Output()

def get_fasta_mask(fasta_path):
    seq = ""
    with open(fasta_path) as f:
        for line in f:
            if line.startswith(">H") or line.startswith(">original_nanobody"):
                continue
            if line.startswith(">"):
                break
            seq += line.strip()
    return [i for i, aa in enumerate(seq) if aa.upper() == 'X']

def on_run_button_clicked(b):
    with output_area:
        clear_output(wait=True)
        print("Starting process...")
        if not upload_pdbs.value:
            print("❌ Please upload the PIPPack ZIP (with the PDBs and cdrs_annotation.json).")
            return

        # optional FASTA fallback
        x_indices = []
        if upload_fasta.value:
            fasta_info = list(upload_fasta.value.values())[0]
            fasta_name = list(upload_fasta.value.keys())[0]
            with open(fasta_name, 'wb') as f:
                f.write(fasta_info['content'])
            x_indices = get_fasta_mask(fasta_name)
            print(f"FASTA fallback mask: {len(x_indices)} 'X' residues.")

        # extract PDBs AND the cdrs_annotation.json
        input_dir = "data/pdbs"
        os.makedirs(input_dir, exist_ok=True)
        for f in glob.glob(f"{input_dir}/*"):
            try: os.remove(f)
            except: pass

        for fn, finfo in upload_pdbs.value.items():
            path = os.path.join(input_dir, fn)
            with open(path, 'wb') as f:
                f.write(finfo['content'])
            if fn.endswith('.zip'):
                with zipfile.ZipFile(path, 'r') as zip_ref:
                    for member in zip_ref.namelist():
                        basename = os.path.basename(member)
                        if not basename or basename.startswith("._") or "__MACOSX" in member:
                            continue
                        if basename.endswith(".pdb"):
                            if basename.startswith("temp_") or basename.startswith("refined_"):
                                continue
                            with open(os.path.join(input_dir, basename), 'wb') as out_f:
                                out_f.write(zip_ref.read(member))
                        elif basename == "cdrs_annotation.json" or basename.endswith(".json"):
                            with open(os.path.join(input_dir, "cdrs_annotation.json"), 'wb') as out_f:
                                out_f.write(zip_ref.read(member))
                os.remove(path)
            elif fn.endswith(".json"):
                os.replace(path, os.path.join(input_dir, "cdrs_annotation.json"))

        pdbs = glob.glob(f"{input_dir}/*.pdb")
        has_json = os.path.exists(os.path.join(input_dir, "cdrs_annotation.json"))
        print(f"✅ Extracted {len(pdbs)} PDBs. cdrs_annotation.json present: {has_json}")
        if len(pdbs) == 0:
            print("❌ No valid PDBs found in the uploaded zip.")
            return

        runner_code = r"""import sys, os, glob, json
import pandas as pd
import numpy as np
import Bio.PDB
import zipfile

sys.path.append(os.path.abspath("."))
import antifold.main
import antifold.antiscripts

INPUT_DIR = "__INPUT_DIR__"
USE_JSON  = __USE_JSON__
FASTA_MASK = __MASK_INDICES__   # fallback: 0-based nanobody CDR indices from the FASTA

# --- load per-design CDR annotations (keys may carry the .pdb extension) ---
annotations = {}
annot_file = os.path.join(INPUT_DIR, "cdrs_annotation.json")
if USE_JSON and os.path.exists(annot_file):
    with open(annot_file) as f:
        raw = json.load(f)
    for k, v in raw.items():
        annotations[k] = v
        annotations[os.path.splitext(k)[0]] = v
    print(f"Loaded CDR annotations for {len(raw)} design(s) from cdrs_annotation.json")
else:
    print("No cdrs_annotation.json available -> using the FASTA fallback mask.")

def indices_from_bounds(bounds):
    idx = []
    for cdr in ("CDR1", "CDR2", "CDR3"):
        s, e = bounds[cdr]
        idx.extend(range(s - 1, e))   # 1-based inclusive -> 0-based
    return idx

# globals updated before each design
CURRENT_MASK = set(FASTA_MASK)
CURRENT_NB_CHAIN = "H"

orig_get_imgt_mask = antifold.antiscripts.get_imgt_mask
def custom_imgt_mask(df, imgt_regions):
    if "CUSTOM_FASTA" in imgt_regions:
        mask = np.zeros(len(df), dtype=bool)
        chain_col = "pdb_chain" if "pdb_chain" in df.columns else None
        nb_idx = 0
        for i in range(len(df)):
            is_nb = (df.iloc[i][chain_col] == CURRENT_NB_CHAIN) if chain_col else True
            if is_nb:
                if nb_idx in CURRENT_MASK:
                    mask[i] = True
                nb_idx += 1
        return mask
    return orig_get_imgt_mask(df, imgt_regions)
antifold.antiscripts.get_imgt_mask = custom_imgt_mask

# --- enumerate the design complexes ---
parser = Bio.PDB.PDBParser(QUIET=True)
designs = []
for p in sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdb"))):
    bn = os.path.basename(p)
    if bn.startswith("._") or bn.startswith("temp_") or bn.startswith("refined_"):
        continue
    try:
        st = parser.get_structure("s", p)
        chs = list(st[0].get_chains())
        if not chs:
            continue
        ids = [c.id for c in chs]
        nb = "H" if "H" in ids else min(st[0], key=lambda c: len(list(c.get_residues()))).id
        ag = next((c for c in ids if c != nb), "")
        designs.append((os.path.splitext(bn)[0], nb, ag))
    except Exception as e:
        print(f"skip {p}: {e}")

if not designs:
    print("ERROR: No valid design PDBs found.")
    sys.exit(1)

print(f"Running AntiFold on {len(designs)} design complex(es)...")
model = antifold.antiscripts.load_model()

all_res = {}
for pdb_id, nb, ag in designs:
    bounds = annotations.get(pdb_id) or annotations.get(pdb_id + ".pdb")
    if bounds:
        CURRENT_MASK = set(indices_from_bounds(bounds)); src = "JSON"
    else:
        CURRENT_MASK = set(FASTA_MASK); src = "FASTA-fallback"
    CURRENT_NB_CHAIN = nb
    print(f"  {pdb_id}: nanobody='{nb}', antigen='{ag}', CDR mask = {len(CURRENT_MASK)} residues [{src}]")
    if not CURRENT_MASK:
        print(f"    WARNING: empty CDR mask for {pdb_id}; nothing would be redesigned. Skipping.")
        continue
    csv1 = pd.DataFrame([{"pdb": pdb_id, "Hchain": nb, "antigen_chain": ag}],
                        columns=["pdb", "Hchain", "antigen_chain"])
    try:
        res = antifold.main.sample_pdbs(
            model=model, pdbs_csv_or_dataframe=csv1, out_dir="output", pdb_dir=INPUT_DIR,
            regions_to_mutate=["CUSTOM_FASTA"], sample_n=__SAMPLE_N__,
            sampling_temp=__SAMPLING_TEMP__, seed=__SEED__, save_flag=True,
            nanobody_mode=True, custom_chain_mode=True)
        all_res.update(res)
    except Exception as e:
        print(f"    ERROR on {pdb_id}: {e}")

if not all_res:
    print("ERROR: AntiFold produced no sequences for any design.")
    sys.exit(1)

print("\n--- Filtering Top 1 Sequences ---")
out_zip = "AntiFold_Best_Designs.zip"
n_written = 0
with zipfile.ZipFile(out_zip, "w") as zf:
    with open("top1_sequences.fasta", "w") as fa:
        for pdb_id, out_dict in all_res.items():
            best_seq, best_score = "", -float("inf")
            for seq_id, rec in out_dict["sequences"].items():
                desc = rec.description
                sc = [x for x in desc.split(",") if "score=" in x and "global_score" not in x]
                if not sc:
                    continue
                s = float(sc[0].split("=")[1])
                if s > best_score:
                    best_score, best_seq = s, str(rec.seq)
            if best_seq:
                fa.write(f">{pdb_id} score={best_score:.3f}\n{best_seq}\n")
                n_written += 1
    zf.write("top1_sequences.fasta")
    for p in sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdb"))):
        bn = os.path.basename(p)
        if not (bn.startswith("._") or bn.startswith("temp_") or bn.startswith("refined_")):
            zf.write(p, bn)
print(f"AntiFold finished. Wrote {n_written} top sequence(s).")"""

        runner_code = runner_code.replace("__MASK_INDICES__", str(x_indices))
        runner_code = runner_code.replace("__SAMPLE_N__", str(num_sequences.value))
        runner_code = runner_code.replace("__SAMPLING_TEMP__", str(temperature.value))
        runner_code = runner_code.replace("__SEED__", str(seed_value.value))
        runner_code = runner_code.replace("__USE_JSON__", "True" if use_json_annotation.value else "False")
        runner_code = runner_code.replace("__INPUT_DIR__", input_dir)

        with open("run_antifold_custom.py", "w") as f:
            f.write(runner_code)

        print("\n🚀 Running AntiFold... this may take a few minutes.")
        try:
            result = subprocess.run(["/usr/local/envs/antifold/bin/python", "run_antifold_custom.py"],
                                    capture_output=True, text=True, check=True)
            print("\n--- AntiFold Output ---")
            print(result.stdout)
            if not os.path.exists("AntiFold_Best_Designs.zip"):
                print("❌ No output zip was produced — see the log above.")
                return
            print("✅ AntiFold completed. Downloading AntiFold_Best_Designs.zip ...")
            files.download("AntiFold_Best_Designs.zip")
        except subprocess.CalledProcessError as e:
            print("\n❌ ERROR: AntiFold failed!")
            print(e.stdout)
            print(e.stderr)

run_button.on_click(on_run_button_clicked)
display(run_button, output_area)


---
### ✅ Step complete — disconnect before the next notebook

To free the GPU/CPU for the next step, **disconnect and delete this runtime**:
`Runtime → Disconnect and delete runtime` (or `Manage sessions → Terminate`).

⬅️ **[Back to the main workflow](https://biochorl.github.io/Nanobody_de_novo_design/)**
